In [4]:
# unusual_query_ml_pipeline.py

import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score



In [5]:
# -------------------------------
# 1. LOAD DATASET
# -------------------------------

data_path = "unusual_query_detection_dataset_1200_rows.csv"
df = pd.read_csv(data_path)

print("Dataset Loaded")
print("Shape:", df.shape)



Dataset Loaded
Shape: (1230, 13)


In [6]:

# -------------------------------
# 2. PREPARE FEATURES
# -------------------------------

X = df.drop(columns=['unusual_query_flag','event_id','timestamp'])
y = df['unusual_query_flag']

categorical_cols = ['user_id','table_accessed','query_type']
numerical_cols = [col for col in X.columns if col not in categorical_cols]




In [7]:
# -------------------------------
# 3. PREPROCESSING PIPELINE
# -------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', 'passthrough', numerical_cols)
    ]
)


In [8]:


# -------------------------------
# 4. MODEL
# -------------------------------

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)


pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', model)
])


In [9]:


# -------------------------------
# 5. TRAIN TEST SPLIT
# -------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [10]:


# -------------------------------
# 6. TRAIN MODEL
# -------------------------------

print("\nTraining model...")

pipeline.fit(X_train, y_train)

print("Training complete!")



Training model...
Training complete!


In [11]:


# -------------------------------
# 7. EVALUATE MODEL
# -------------------------------

predictions = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("\nModel Evaluation")
print("-----------------")
print("Accuracy:", accuracy)
print("\nClassification Report:\n")
print(classification_report(y_test, predictions))



Model Evaluation
-----------------
Accuracy: 1.0

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       211
           1       1.00      1.00      1.00        35

    accuracy                           1.00       246
   macro avg       1.00      1.00      1.00       246
weighted avg       1.00      1.00      1.00       246



In [12]:


# -------------------------------
# 8. SAVE MODEL
# -------------------------------

model_path = "unusual_query_detector.pkl"

joblib.dump(pipeline, model_path)

print("\nModel saved to:", model_path)




Model saved to: unusual_query_detector.pkl


In [13]:

# -------------------------------
# 9. LOAD MODEL
# -------------------------------

print("\nReloading model...")

loaded_model = joblib.load(model_path)

print("Model loaded successfully!")



Reloading model...
Model loaded successfully!


In [14]:


# -------------------------------
# 10. PREDICT NEW DATA
# -------------------------------

new_query = pd.DataFrame([
    {
    "user_id": "user_25",
    "table_accessed": "financial_records",
    "query_type": "SELECT",
    "rows_returned": 1500,
    "after_hours_access": 1,
    "sensitive_data_access": 1,
    "failed_login_attempt": 0,
    "total_queries_session": 45,
    "data_exfiltration_pattern": 1,
    "user_timeline_step": 0 # Added missing column with a default value
    }
])


prediction = loaded_model.predict(new_query)

print("\nPrediction for new query:")

if prediction[0] == 1:
    print("⚠️ Unusual Query Detected")
else:
    print("✅ Normal Query")



Prediction for new query:
⚠️ Unusual Query Detected
